# AutoRepairEstimator — YOLOv8-seg training on Google Colab

Готовый пайплайн для обучения моделей `parts` (детали) и `damages` (повреждения).

**Перед запуском:** локально сгенерировать и заархивировать датасеты:

```powershell
# в корне репозитория на локальной машине
python scripts/ml/split_dataset.py --images test/images_details/Train --labels test/labels/Train --output test/parts --classes 12 --names door front_fender rear_fender trunk hood roof headlight front_windshield rear_windshield side_window wheel bumper --ratios 0.8 0.1 0.1 --seed 42
python scripts/ml/split_dataset.py --images test/0-604damages/images/Train --labels test/0-604damages/labels/Train --output test/damages --classes 8 --names scratch dent paint_chip rust crack broken_glass flat_tire broken_headlight --ratios 0.8 0.1 0.1 --seed 42 --oversample flat_tire=6 rust=3
Compress-Archive -Path test\parts -DestinationPath test\parts.zip
Compress-Archive -Path test\damages -DestinationPath test\damages.zip
```

Загрузите `parts.zip` и `damages.zip` на Google Drive (в папку `ML/AutoRepairEstimator/`).

## 0. Проверка GPU

Runtime → Change runtime type → Hardware accelerator → **GPU (T4 или A100)**.

In [ ]:
!nvidia-smi

## 1. Установка зависимостей

`ultralytics` — пакет YOLOv8. Pin-ить версию не нужно, API стабильный.

In [ ]:
!pip install -q ultralytics

## 2. Монтирование Google Drive и распаковка датасетов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Отредактируйте пути под свою структуру в Drive
DRIVE_ROOT = '/content/drive/MyDrive/ML/AutoRepairEstimator'

!unzip -q -o {DRIVE_ROOT}/parts.zip -d /content/
!unzip -q -o {DRIVE_ROOT}/damages.zip -d /content/
!ls /content/parts /content/damages

## 3. (Опционально) Убрать старый `path:` из data.yaml

Актуальный `split_dataset.py` **не пишет** ключ `path:` — корень датасета = папка, где лежит `data.yaml` (так резолвит Ultralytics 8.4+). Если ваш zip собран со старой версией скрипта и в yaml осталась строка `path: D:\...` или `/mnt/d/...`, выполните ячейку ниже — иначе Colab будет искать картинки по Windows-пути.

In [ ]:
import re
from pathlib import Path

def strip_stale_dataset_path(yaml_path: str) -> None:
    p = Path(yaml_path)
    if not p.exists():
        print(f"skip (not found): {yaml_path}")
        return
    content = p.read_text(encoding="utf-8")
    new = re.sub(r"^path:\s*.*\n", "", content, flags=re.M)
    if new != content:
        p.write_text(new, encoding="utf-8")
        print(f"removed stale `path:` from {p}")
    else:
        print(f"OK (no `path:` or already portable): {p}")

# Подставьте свои пути, если распаковали не в /content/parts
for y in ("/content/parts/data.yaml", "/content/damages/data.yaml", "/content/details/data.yaml"):
    strip_stale_dataset_path(y)

## 4. Загрузить тренировочные скрипты из репозитория

Лучший способ — клонировать репозиторий. Если он приватный, используйте свой токен или заранее выложите `scripts/ml/` на Drive отдельным архивом.

In [ ]:
# Вариант A: клон публичного/доступного репозитория
# !git clone https://github.com/<you>/AutoRepairEstimator.git /content/repo
# %cd /content/repo

# Вариант B: если scripts/ml/ лежат на Drive
!mkdir -p /content/repo/scripts/ml
!cp {DRIVE_ROOT}/scripts/ml/*.py /content/repo/scripts/ml/
%cd /content/repo
!ls scripts/ml/

## 5. Обучение модели деталей (`parts`)

Гиперпараметры зашиты в `scripts/ml/train_config.py` (AdamW, lr=0.001, cls_pw, copy_paste, mosaic). На T4 одна эпоха ≈ 60–90 сек, 150 эпох ≈ 3 часа. Patience=30 обычно останавливает раньше.

Для быстрой проверки пайплайна (≤5 минут) уменьшите epochs до 5.

In [ ]:
!python scripts/ml/train_parts.py \
    --data /content/parts/data.yaml \
    --device 0 \
    --batch 8 \
    --project /content/runs \
    --name parts_v1

## 6. Обучение модели повреждений (`damages`)

Использует отдельный конфиг с усиленным cls_pw=1.0 (inverse-frequency веса классов) и copy_paste=0.3 — это компенсирует сильный дисбаланс `scratch` vs остальные. Параметр fl_gamma в Ultralytics 8.4+ удалён.

In [ ]:
!python scripts/ml/train_damages.py \
    --data /content/damages/data.yaml \
    --device 0 \
    --batch 8 \
    --project /content/runs \
    --name damages_v1

## 7. Оценка на test-сете

`split='test'` — holdout, который ни обучение, ни early stopping не видели. Это честная оценка.

In [ ]:
from ultralytics import YOLO

for run, data in [
    ('parts_v1', '/content/parts/data.yaml'),
    ('damages_v1', '/content/damages/data.yaml'),
]:
    weights = f'/content/runs/{run}/weights/best.pt'
    print(f'=== {run} ({weights}) ===')
    model = YOLO(weights)
    metrics = model.val(data=data, split='test', plots=True, save_json=False)
    print(f"mAP50 (box):  {metrics.box.map50:.3f}")
    print(f"mAP50-95 (box): {metrics.box.map:.3f}")
    print(f"mAP50 (seg):  {metrics.seg.map50:.3f}")
    print(f"mAP50-95 (seg): {metrics.seg.map:.3f}")
    print()

## 8. Поиск оптимального порога уверенности

Ищем порог, при котором **минимальный** per-class F1 максимален (bottleneck-оптимизация, не среднее). Эти значения потом идут в `backend/domain/value_objects/ml_thresholds.py`.

In [ ]:
import numpy as np
from ultralytics import YOLO

def find_best_threshold(weights: str, data_yaml: str, split: str = 'val'):
    results = []
    model = YOLO(weights)
    for thr in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
        r = model.val(data=data_yaml, conf=thr, split=split, plots=False, verbose=False)
        f1_per_class = r.box.f1.tolist()
        results.append((thr, min(f1_per_class), np.mean(f1_per_class), f1_per_class))
        print(f'  conf={thr}: min_f1={min(f1_per_class):.3f}, mean_f1={np.mean(f1_per_class):.3f}')
    best = max(results, key=lambda x: x[1])
    print(f'  → best threshold: {best[0]} (min_f1={best[1]:.3f})')
    return best

print('=== parts ===')
find_best_threshold('/content/runs/parts_v1/weights/best.pt', '/content/parts/data.yaml')
print()
print('=== damages ===')
find_best_threshold('/content/runs/damages_v1/weights/best.pt', '/content/damages/data.yaml')

## 9. Сохранение весов и результатов обратно на Drive

In [ ]:
from datetime import datetime
import shutil

stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
out_dir = f'{DRIVE_ROOT}/runs/{stamp}'
!mkdir -p {out_dir}

# best.pt + results.csv + confusion matrix + PR curves
for run in ['parts_v1', 'damages_v1']:
    src = f'/content/runs/{run}'
    dst = f'{out_dir}/{run}'
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Saved {src} -> {dst}')

!ls -la {out_dir}

## 10. Gate-метрики перед выкаткой в прод

Минимум для адекватной модели:

| Метрика | Parts | Damages |
| --- | --- | --- |
| `mAP50(M)` на val (avg) | ≥ 0.65 | ≥ 0.45 |
| Худший per-class `mAP50(M)` | ≥ 0.30 | ≥ 0.20 |
| `mAP50(B)` на test | ≥ 0.60 | ≥ 0.40 |

Если ниже — **в прод не катим**, возвращаемся в CVAT по правилам из `docs/ml/annotation_and_training_guide.md`.